# 00 — Data Preparation

Builds all dataset files used by every subsequent notebook in this repository.
All outputs are written to `paper2/data/` on Google Drive.

## Split design

| Label | Plans per domain | Total | Purpose |
|---|---|---|---|
| `fewshot_invalid` | 5 invalid | 25 | FlanT5 stepwise training (01a, 01b); 1 plan per domain used as GPT few-shot invalid example |
| `fewshot_valid` | 5 valid | 25 | FlanT5 direct fine-tuning training (01c, FT50); 1 plan per domain used as GPT few-shot valid example |
| `test` | all remaining | 1,534 | Evaluation in all conditions |

Plans are selected by verified parseability: the manifest guarantees exactly
5 parseable invalid + 5 parseable valid plans per domain, with zero overlap
between fewshot and test sets.

## Files written to `paper2/data/`

| File | Contents |
|---|---|
| `split_manifest.json` | plan_id → split label for every plan in every domain |
| `planbench_stepwise_fewshot.jsonl` | Stepwise action records for 25 fewshot_invalid plans (FlanT5 training) |
| `planbench_stepwise_test.jsonl` | Stepwise action records for all test plans (evaluation) |
| `planbench_direct_fewshot.jsonl` | Direct plan records for 1 valid + 1 invalid plan per domain (GPT 2-shot examples) |
| `planbench_direct_test.jsonl` | Direct plan records for all test plans (evaluation) |
| `dataset_stats.json` | Per-domain plan and record counts |

## Mystery domain vocabulary abstraction

The Mystery and Mystery-3 domains use semantically meaningful action and
relation terms (e.g., `attack`, `Province`, `Craves`) that could allow a
model to exploit real-world knowledge rather than checking preconditions
logically. Before any model interaction, all such terms are replaced with
abstract tokens. This abstraction is applied consistently across all
prompts, training, and evaluation notebooks.

| Original | Abstract | Original | Abstract |
|---|---|---|---|
| attack / Attack | action1 / Action1 | Province | Rel1 |
| succumb / Succumb | action2 / Action2 | Planet | Rel2 |
| overcome / Overcome | action3 / Action3 | Harmony | Rel3 |
| feast / Feast | action4 / Action4 | Pain | Rel4 |
| | | Craves | Rel5 |

## 1. Setup

Mount Google Drive and install required packages.
All paths are relative to `ROOT = paper2/` — no external folder references.
The `data/` subdirectory is created automatically if it does not exist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U datasets huggingface_hub

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via HF_TOKEN secret')
except Exception:
    from huggingface_hub import login
    login()

In [ ]:
import os, re, json, random
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Set, Tuple

from datasets import load_dataset

SEED = 42
random.seed(SEED)

# ── paper2 root ─────────────────────────────────────────────────────────────────
ROOT     = '/content/drive/MyDrive/paper_codes/paper2'
DATA_DIR = f'{ROOT}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# ── Output paths ──────────────────────────────────────────────────────────────
MANIFEST_PATH    = os.path.join(DATA_DIR, 'split_manifest.json')
SW_FEWSHOT_PATH  = os.path.join(DATA_DIR, 'planbench_stepwise_fewshot.jsonl')
DI_FEWSHOT_PATH  = os.path.join(DATA_DIR, 'planbench_direct_fewshot.jsonl')
SW_TEST_PATH     = os.path.join(DATA_DIR, 'planbench_stepwise_test.jsonl')
DI_TEST_PATH     = os.path.join(DATA_DIR, 'planbench_direct_test.jsonl')
STATS_PATH       = os.path.join(DATA_DIR, 'dataset_stats.json')

# ── Dataset config ────────────────────────────────────────────────────────────
HF_DATASET = 'tasksource/planbench'
HF_SUBSET  = 'task_3_plan_verification'

DOMAIN_MAP = {
    'mystery_blocksworld'  : 'mystery',
    'mystery_blocksworld_3': 'mystery_3',
    'blocksworld'          : 'blocksworld',
    'blocksworld_3'        : 'blocksworld_3',
    'logistics'            : 'logistics',
}
DOMAINS = list(DOMAIN_MAP.values())

# ── Split sizes ───────────────────────────────────────────────────────────────
N_FEWSHOT_INVALID_PER_DOMAIN = 5   # for FlanT5 training + 1 GPT invalid example
N_FEWSHOT_VALID_PER_DOMAIN   = 5   # for FlanT5_Direct_FT50 training + 1 GPT valid example

print(f'ROOT     : {ROOT}')
print(f'DATA_DIR : {DATA_DIR}')
print(f'fewshot_invalid : {N_FEWSHOT_INVALID_PER_DOMAIN}/domain = '
      f'{N_FEWSHOT_INVALID_PER_DOMAIN * len(DOMAINS)} total')
print(f'fewshot_valid   : {N_FEWSHOT_VALID_PER_DOMAIN}/domain = '
      f'{N_FEWSHOT_VALID_PER_DOMAIN * len(DOMAINS)} total')

## 2. Mystery Domain Vocabulary Abstraction

The Mystery and Mystery-3 domains use semantically meaningful terms that
could allow a model to exploit world knowledge rather than performing logical
inference from preconditions. All such terms are replaced with abstract tokens
before any prompt is constructed.

The `abstract_mystery(text)` function applies whole-word, case-sensitive
substitution using the mapping defined in `MYSTERY_FULL_MAP`. It is called
on all rules, state facts, actions, and goal descriptions for Mystery domains
before any record is built. The mapping is the same across all notebooks.

In [ ]:
MYSTERY_ACTION_MAP = {
    'attack':'action1','succumb':'action2','overcome':'action3','feast':'action4',
    'Attack':'Action1','Succumb':'Action2','Overcome':'Action3','Feast':'Action4',
}
MYSTERY_REL_MAP = {
    'Province':'Rel1','province':'rel1','Planet':'Rel2','planet':'rel2',
    'Harmony':'Rel3','harmony':'rel3','Pain':'Rel4','pain':'rel4',
    'Craves':'Rel5','craves':'rel5',
}
MYSTERY_FULL_MAP = {**MYSTERY_ACTION_MAP, **MYSTERY_REL_MAP}

def abstract_mystery(text: str) -> str:
    result = text
    for original, replacement in MYSTERY_FULL_MAP.items():
        result = re.sub(r'\b' + re.escape(original) + r'\b', replacement, result)
    return result

# Quick tests
assert abstract_mystery('attack object a')   == 'action1 object a'
assert abstract_mystery('Province object a') == 'Rel1 object a'
assert abstract_mystery('province object a') == 'rel1 object a'
assert abstract_mystery('Craves object a')   == 'Rel5 object a'
print('Abstraction: OK')

## 3. Prompt Builders

Two prompt formats are used throughout this repository:

**Stepwise format** (`build_stepwise_prompt`): presents domain rules and the
current world state as premises, and a single action as the conclusion.
The model outputs `A` (action executable — all preconditions satisfied) or
`B` (action not executable — precondition violated). This format is
structurally identical to FOLIO logical reasoning, enabling transfer learning.

**Direct format** (`build_direct_prompt`): presents the full plan (all rules,
initial state, goal, and action sequence) in a single prompt and asks for
a single `A` (valid) or `B` (invalid) verdict. Used for direct fine-tuning
baselines (01c) and GPT direct evaluation (03, 04).

In [ ]:
def build_stepwise_prompt(rules, state_facts, conclusion):
    premises  = rules + state_facts
    prem_block = '\n'.join('- ' + p.strip().rstrip('.') + '.' for p in premises if p.strip())
    return (
        'Task: Determine whether the conclusion is entailed or contradicted given the premises.\n'
        'Premises:\n' + prem_block + '\n\n'
        'Conclusion:\n' + conclusion.strip() + '\n\n'
        'Output format: Answer A if the conclusion is entailed, B if it is contradicted.\n'
        'Answer:'
    )

def build_direct_prompt(rules, init_facts, goal_facts, actions):
    rules_block = '\n'.join('- ' + r.strip().rstrip('.') + '.' for r in rules if r.strip())
    state_block = '\n'.join('- ' + f.strip().rstrip('.') + '.' for f in init_facts if f.strip())
    goal_block  = '\n'.join('- ' + g.strip().rstrip('.') + '.' for g in goal_facts if g.strip())
    plan_block  = '\n'.join(f'{i+1}. {a.strip()}' for i, a in enumerate(actions))
    return (
        'Task: Determine whether the following plan is valid given the rules and initial state.\n\n'
        'Rules:\n' + rules_block + '\n\n'
        'Initial state:\n' + state_block + '\n\n'
        'Goal:\n' + goal_block + '\n\n'
        'Plan:\n' + plan_block + '\n\n'
        'Output format: Answer A if the plan is valid, B if it is not.\n'
        'Answer:'
    )

print('Prompt builders: OK')

## 4. Parsers

These functions extract structured fields from raw HuggingFace PlanBench records.

- `parse_rules(query, domain)` — extracts domain action precondition/effect rules;
  filters out header lines and applies Mystery abstraction for Mystery domains
- `parse_query(query, domain)` — extracts initial state text, goal text, and
  action list from the last `[STATEMENT]` block in the query
- `parse_actions(text)` — extracts the action sequence between `[PLAN]` and `[PLAN END]`
- `parse_ground_truth(gt)` — extracts the validity label and failure mode:
  - `label=1, fail_reason='valid'` for valid plans
  - `label=0, fail_reason='action_fail', failing_step=n` (1-indexed) for plans
    that fail because action $n$ violates a precondition
  - `label=0, fail_reason='goal_fail', failing_step=None` for plans where all
    actions execute but the final state does not satisfy the goal
  - Records with missing ground truth are excluded from all datasets

In [ ]:
def parse_rules(query, domain):
    m = re.search(
        r'(?:I have the following restrictions on my actions:|'
        r'Here are the actions that can be performed:)(.*?)(?=\[STATEMENT\])',
        query, re.DOTALL | re.I)
    if not m: return []
    lines = [l.strip() for l in m.group(1).strip().split('\n') if l.strip()]
    lines = [l for l in lines if not re.match(r'for example[,.]', l, re.I)]
    bw_headers = {'pick up a block','unstack a block from on top of another block',
                  'put down a block','stack a block on top of another block'}
    mystery_headers = {'attack object','feast object from another object',
                       'succumb object','overcome object from another object'}
    skip = bw_headers | mystery_headers
    lines = [l for l in lines if l.lower().rstrip('.') not in skip]
    lines = [l for l in lines if len(l) > 10]
    if domain in ('mystery','mystery_3'):
        lines = [abstract_mystery(l) for l in lines]
    return lines

def parse_query(query, domain):
    stmt_positions = [m.start() for m in re.finditer(r'\[STATEMENT\]', query)]
    if not stmt_positions: return None
    last_stmt = query[stmt_positions[-1]:]
    im = re.search(r'As initial conditions I have that,?\s*(.*?)\s*My goal',
                   last_stmt, re.DOTALL | re.I)
    gm = re.search(r'My goal is to have that\s*(.*?)\s*My plan',
                   last_stmt, re.DOTALL | re.I)
    if not im or not gm: return None
    actions = parse_actions(last_stmt)
    if not actions: return None
    return {'init_text': im.group(1).strip(), 'goal_text': gm.group(1).strip(),
            'actions': actions, 'rules': parse_rules(query, domain)}

def parse_actions(text):
    m = re.search(r'\[PLAN\]\s*(.*?)\s*\[PLAN END\]', text, re.DOTALL)
    if not m: return []
    return [a.strip() for a in m.group(1).strip().split('\n') if a.strip()]

def parse_ground_truth(gt):
    if gt is None: return {'label':0,'failing_step':None,'fail_reason':'missing'}
    gt = str(gt).strip()
    if not gt: return {'label':0,'failing_step':None,'fail_reason':'missing'}
    if re.search(r'the above plan is valid', gt, re.I):
        return {'label':1,'failing_step':None,'fail_reason':'valid'}
    m = re.search(r'action at step (\d+)', gt, re.I)
    if m: return {'label':0,'failing_step':int(m.group(1)),'fail_reason':'action_fail'}
    return {'label':0,'failing_step':None,'fail_reason':'goal_fail'}

print('Parsers: OK')

## 5. Conclusion Builders

In the stepwise format, each action is rephrased as a natural language
hypothesis (conclusion) to be tested against the current state premises.
These functions convert action strings to conclusions for each domain:

- `bw_action_to_conclusion` — Blocksworld actions (pick up, put down, stack, unstack)
- `mystery_action_to_conclusion` — Mystery actions using abstract vocabulary (action1..action4)
- `logistics_action_to_conclusion` — Logistics actions (load, unload, drive, fly)

Actions that cannot be parsed into a conclusion are skipped. For valid steps,
the state is still updated deterministically so that subsequent steps
receive correct premises.

In [ ]:
def bw_action_to_conclusion(action):
    a = action.strip().lower()
    m = re.match(r'unstack (?:the )?(\w+) block from (?:on top of )?(?:the )?(\w+) block', a)
    if m: return f'The {m.group(1)} block can be unstacked from on top of the {m.group(2)} block.'
    m = re.match(r'pick up (?:the )?(\w+) block', a)
    if m: return f'The {m.group(1)} block can be picked up.'
    m = re.match(r'put down (?:the )?(\w+) block', a)
    if m: return f'The {m.group(1)} block can be put down.'
    m = re.match(r'stack (?:the )?(\w+) block on (?:top of )?(?:the )?(\w+) block', a)
    if m: return f'The {m.group(1)} block can be stacked on top of the {m.group(2)} block.'
    return None

def mystery_action_to_conclusion(action):
    a = abstract_mystery(action.strip().lower())
    m = re.match(r'action(\d) object (\w+)$', a)
    if m: return f'Action{m.group(1)} can be performed on object {m.group(2)}.'
    m = re.match(r'action(\d) object (\w+) from object (\w+)$', a)
    if m: return f'Action{m.group(1)} can be performed on object {m.group(2)} from object {m.group(3)}.'
    return None

def logistics_action_to_conclusion(action):
    a = action.strip().lower()
    m = re.match(r'load (\S+) into (\S+) at (\S+)', a)
    if m: return f'{m.group(1).capitalize()} can be loaded into {m.group(2)} at {m.group(3)}.'
    m = re.match(r'unload (\S+) from (\S+) at (\S+)', a)
    if m: return f'{m.group(1).capitalize()} can be unloaded from {m.group(2)} at {m.group(3)}.'
    m = re.match(r'drive (\S+) from (\S+) to (\S+) in (\S+)', a)
    if m: return f'{m.group(1).capitalize()} can be driven from {m.group(2)} to {m.group(3)}.'
    m = re.match(r'fly (\S+) from (\S+) to (\S+)', a)
    if m: return f'{m.group(1).capitalize()} can be flown from {m.group(2)} to {m.group(3)}.'
    return None

def action_to_conclusion(action, domain):
    if domain in ('blocksworld','blocksworld_3'): return bw_action_to_conclusion(action)
    if domain in ('mystery','mystery_3'):         return mystery_action_to_conclusion(action)
    if domain == 'logistics':                     return logistics_action_to_conclusion(action)
    raise ValueError(f'Unknown domain: {domain}')

print('Conclusion builders: OK')

## 6. State Classes and Deterministic Updaters

Each domain has a dataclass representing the world state and a deterministic
state transition function. These are used during stepwise record construction
to compute the correct world state facts for each action's premises.

- **Blocksworld / Blocksworld-3**: `BWState` + `bw_apply` — tracks block
  positions, clear status, and hand state
- **Mystery / Mystery-3**: `MysteryState` + `mystery_apply` — tracks abstract
  relational facts (rel1..rel5) with exact action precondition/effect semantics
- **Logistics**: `LogisticsState` + `logistics_apply` — tracks entity locations,
  airports, and city mappings

The state is only updated when an action is labelled `A` (valid). This
prevents error propagation: a mispredicted step does not corrupt subsequent
premises. `check_goal(state, goal_facts)` compares the final state to the
goal specification at the end of the pipeline.

In [ ]:
@dataclass
class BWState:
    on: List[Tuple[str,str]] = field(default_factory=list)
    on_table: List[str]      = field(default_factory=list)
    clear: List[str]         = field(default_factory=list)
    holding: Optional[str]   = None
    hand_empty: bool          = True

    def to_facts(self):
        f = []
        for x,y in self.on:     f.append(f'The {x} block is on top of the {y} block.')
        for x in self.on_table: f.append(f'The {x} block is on the table.')
        for x in self.clear:    f.append(f'The {x} block is clear.')
        if self.holding:        f.append(f'You are holding the {self.holding} block.')
        if self.hand_empty:     f.append('The hand is empty.')
        return f

    def fact_set(self):
        return frozenset(f.lower().rstrip('.') for f in self.to_facts())

    def copy(self):
        return BWState(on=list(self.on), on_table=list(self.on_table),
                       clear=list(self.clear), holding=self.holding,
                       hand_empty=self.hand_empty)

def bw_parse_facts(text):
    parts = re.split(r',|\band\b', text.lower())
    on,on_table,clear=[],[],[]; holding=None; hand_empty=False
    for p in parts:
        t=p.strip().rstrip('.')
        m=re.search(r'the (\w+) block is on top of the (\w+) block',t)
        if m: on.append((m.group(1),m.group(2))); continue
        m=re.search(r'the (\w+) block is on the table',t)
        if m: on_table.append(m.group(1)); continue
        m=re.search(r'the (\w+) block is clear',t)
        if m: clear.append(m.group(1)); continue
        m=re.search(r'holding (?:the )?(\w+) block',t)
        if m: holding=m.group(1); continue
        if 'hand is empty' in t: hand_empty=True
    return BWState(on=on,on_table=on_table,clear=clear,holding=holding,hand_empty=hand_empty)

def bw_parse_goal(text):
    parts=re.split(r',|\band\b',text.lower()); facts=[]
    for p in parts:
        t=p.strip().rstrip('.')
        m=re.search(r'the (\w+) block is on top of the (\w+) block',t)
        if m: facts.append(f'the {m.group(1)} block is on top of the {m.group(2)} block'); continue
        m=re.search(r'the (\w+) block is on the table',t)
        if m: facts.append(f'the {m.group(1)} block is on the table'); continue
        m=re.search(r'the (\w+) block is clear',t)
        if m: facts.append(f'the {m.group(1)} block is clear')
    return facts

def bw_apply(state, action):
    s=state.copy(); a=action.strip().lower()
    m=re.match(r'unstack (?:the )?(\w+) block from (?:on top of )?(?:the )?(\w+) block',a)
    if m:
        X,Y=m.group(1),m.group(2)
        s.on=[(a2,b2) for a2,b2 in s.on if not(a2==X and b2==Y)]
        s.clear=[c for c in s.clear if c!=X]
        s.hand_empty=False; s.holding=X
        if Y not in s.clear: s.clear.append(Y)
        return s
    m=re.match(r'pick up (?:the )?(\w+) block',a)
    if m:
        X=m.group(1); s.on_table=[b for b in s.on_table if b!=X]
        s.clear=[c for c in s.clear if c!=X]; s.hand_empty=False; s.holding=X; return s
    m=re.match(r'put down (?:the )?(\w+) block',a)
    if m:
        X=m.group(1); s.holding=None; s.hand_empty=True
        if X not in s.on_table: s.on_table.append(X)
        if X not in s.clear: s.clear.append(X); return s
    m=re.match(r'stack (?:the )?(\w+) block on (?:top of )?(?:the )?(\w+) block',a)
    if m:
        X,Y=m.group(1),m.group(2); s.holding=None; s.hand_empty=True
        s.clear=[c for c in s.clear if c!=Y]
        if X not in s.clear: s.clear.append(X)
        if (X,Y) not in s.on: s.on.append((X,Y)); return s
    return None

# ── Mystery state ─────────────────────────────────────────────────────────────
@dataclass
class MysteryState:
    facts: Set[str] = field(default_factory=set)
    def to_facts(self):
        return sorted(f.capitalize()+'.' if not f.endswith('.') else f.capitalize() for f in self.facts)
    def fact_set(self): return frozenset(f.lower().rstrip('.') for f in self.facts)
    def copy(self): return MysteryState(facts=set(self.facts))

def mystery_parse_facts(text):
    text=abstract_mystery(text.lower().strip()); parts=re.split(r',|\band\b',text); facts=set()
    for p in parts:
        t=p.strip().rstrip('.')
        if (t=='rel3' or re.match(r'rel[1-4] object \w+',t)
                or re.match(r'object \w+ rel5 object \w+',t)): facts.add(t)
    return MysteryState(facts=facts)

def mystery_parse_goal(text):
    text=abstract_mystery(text.lower().strip()); parts=re.split(r',|\band\b',text)
    return [p.strip().rstrip('.') for p in parts
            if re.match(r'object \w+ rel5 object \w+',p.strip().rstrip('.'))]

def mystery_apply(state, action):
    s=state.copy(); a=abstract_mystery(action.strip().lower())
    m=re.match(r'action1 object (\w+)$',a)
    if m:
        x=m.group(1); s.facts-={f'rel1 object {x}',f'rel2 object {x}','rel3'}
        s.facts.add(f'rel4 object {x}'); return s
    m=re.match(r'action2 object (\w+)$',a)
    if m:
        x=m.group(1); s.facts.discard(f'rel4 object {x}')
        s.facts|={f'rel1 object {x}',f'rel2 object {x}','rel3'}; return s
    m=re.match(r'action3 object (\w+) from object (\w+)$',a)
    if m:
        x,y=m.group(1),m.group(2); s.facts-={f'rel1 object {y}',f'rel4 object {x}'}
        s.facts|={'rel3',f'rel1 object {x}',f'object {x} rel5 object {y}'}; return s
    m=re.match(r'action4 object (\w+) from object (\w+)$',a)
    if m:
        x,y=m.group(1),m.group(2)
        s.facts-={f'object {x} rel5 object {y}',f'rel1 object {x}','rel3'}
        s.facts|={f'rel4 object {x}',f'rel1 object {y}'}; return s
    return None

# ── Logistics state ───────────────────────────────────────────────────────────
@dataclass
class LogisticsState:
    locations: Dict[str,str] = field(default_factory=dict)
    airports: Set[str]       = field(default_factory=set)
    city_map: Dict[str,str]  = field(default_factory=dict)

    def to_facts(self):
        f=[]
        for e,loc in sorted(self.locations.items()):
            f.append(f'{e} is in {loc}.' if any(loc.startswith(v) for v in ('truck','airplane'))
                     else f'{e} is at {loc}.')
        for loc in sorted(self.airports): f.append(f'{loc} is an airport.')
        for loc,city in sorted(self.city_map.items()): f.append(f'{loc} is in city {city}.')
        return f

    def fact_set(self): return frozenset(f.lower().rstrip('.') for f in self.to_facts())
    def copy(self): return LogisticsState(locations=dict(self.locations),
                                          airports=set(self.airports),
                                          city_map=dict(self.city_map))

def logistics_parse_facts(text):
    parts=re.split(r',|\band\b',text.lower().strip())
    locs={}; airports=set(); city_map={}
    for p in parts:
        t=p.strip().rstrip('.')
        if 'is an airport' in t:
            m=re.search(r'(\S+) is an airport',t)
            if m: airports.add(m.group(1))
        elif re.search(r'is in (?:the )?city',t):
            m=re.search(r'(\S+) is in (?:the )?city (\S+)',t)
            if m: city_map[m.group(1)]=m.group(2)
        elif 'is at' in t:
            m=re.search(r'(\S+) is at (\S+)',t)
            if m: locs[m.group(1)]=m.group(2)
        elif 'is in' in t:
            m=re.search(r'(\S+) is in (\S+)',t)
            if m: locs[m.group(1)]=m.group(2)
    return LogisticsState(locations=locs,airports=airports,city_map=city_map)

def logistics_parse_goal(text):
    parts=re.split(r',|\band\b',text.lower().strip()); facts=[]
    for p in parts:
        t=p.strip().rstrip('.')
        if 'is at' in t:
            m=re.search(r'(\S+) is at (\S+)',t)
            if m: facts.append(f'{m.group(1)} is at {m.group(2)}')
        elif 'is in' in t and 'city' not in t:
            m=re.search(r'(\S+) is in (\S+)',t)
            if m: facts.append(f'{m.group(1)} is in {m.group(2)}')
    return facts

def logistics_apply(state, action):
    s=state.copy(); a=action.strip().lower()
    m=re.match(r'load (\S+) into (\S+) at (\S+)',a)
    if m: s.locations[m.group(1)]=m.group(2); return s
    m=re.match(r'unload (\S+) from (\S+) at (\S+)',a)
    if m: s.locations[m.group(1)]=m.group(3); return s
    m=re.match(r'drive (\S+) from (\S+) to (\S+) in (\S+)',a)
    if m: s.locations[m.group(1)]=m.group(3); return s
    m=re.match(r'fly (\S+) from (\S+) to (\S+)',a)
    if m: s.locations[m.group(1)]=m.group(3); return s
    return None

def get_domain_fns(domain):
    if domain in ('blocksworld','blocksworld_3'): return bw_parse_facts,bw_parse_goal,bw_apply
    if domain in ('mystery','mystery_3'): return mystery_parse_facts,mystery_parse_goal,mystery_apply
    if domain=='logistics': return logistics_parse_facts,logistics_parse_goal,logistics_apply
    raise ValueError(f'Unknown domain: {domain}')

def check_goal(state, goal_facts):
    fs=state.fact_set()
    return all(g.lower().rstrip('.')in fs for g in goal_facts)

print('State classes & updaters: OK')

## 7. Load Dataset from HuggingFace

Loads the PlanBench plan verification task (`task_3_plan_verification`)
from `tasksource/planbench`. A valid HuggingFace token is required,
stored as a Colab secret named `HF_TOKEN`.

The dataset is grouped by domain for efficient per-domain processing.
Total: 1,584 plans across 5 domains (Blocksworld, Blocksworld-3,
Mystery, Mystery-3, Logistics).

In [ ]:
print('Loading tasksource/planbench from HuggingFace...')
hf_ds = load_dataset(HF_DATASET, HF_SUBSET, split='train')
print(f'Loaded {len(hf_ds)} total rows')

from collections import defaultdict as dd
domain_groups = dd(list)
for row in hf_ds:
    domain_groups[row['domain']].append(row)

for hf_domain, internal_domain in DOMAIN_MAP.items():
    rows = domain_groups.get(hf_domain, [])
    print(f'  {internal_domain:15s}: {len(rows)} rows')

## 8. Record Builders

**`build_stepwise_for_plan`**: builds one record per action step.

For valid plans: all steps are labelled `A`.
For invalid plans with `action_fail` at step $n$:
- Steps 1 to $n-1$: labelled `A` (preconditions satisfied)
- Step $n$: labelled `B` (precondition violated — this is the failing step)
- Steps after $n$: omitted (the pipeline would halt at step $n$)

For invalid plans with `goal_fail`: all steps are labelled `A` — every
action is individually executable, but the final state does not satisfy
the goal. These plans contribute only A-labelled records to the training set.

Each record contains the full prompt (rules + current state facts as premises,
action as conclusion), the gold label, and metadata including domain, plan_id,
step number, and fail_reason.

**`build_direct_for_plan`**: builds one record for the entire plan.
The prompt contains all rules, initial state, goal, and the numbered action
sequence. Label is `A` (valid) or `B` (invalid).

In [ ]:
def build_stepwise_for_plan(plan_id, row, domain, split_label):
    query = row.get('query','')
    gt    = row.get('ground_truth_plan','')
    parsed_query = parse_query(query, domain)
    parsed_gt    = parse_ground_truth(gt)
    if parsed_query is None: return []
    if parsed_gt['fail_reason'] == 'missing': return []

    label        = parsed_gt['label']
    failing_step = parsed_gt['failing_step']
    fail_reason  = parsed_gt['fail_reason']
    actions      = parsed_query['actions']
    rules        = parsed_query['rules']
    init_text    = parsed_query['init_text']
    goal_text    = parsed_query['goal_text']

    parse_facts_fn, parse_goal_fn, apply_fn = get_domain_fns(domain)

    if domain in ('mystery','mystery_3'):
        state      = parse_facts_fn(abstract_mystery(init_text))
        goal_facts = parse_goal_fn(abstract_mystery(goal_text))
    else:
        state      = parse_facts_fn(init_text)
        goal_facts = parse_goal_fn(goal_text)

    records = []
    for step_idx, action in enumerate(actions):
        step_num = step_idx + 1
        if label == 1:
            step_gold = 'A'; step_gold_int = 1
        else:
            if failing_step is None:
                step_gold = 'A'; step_gold_int = 1
            elif step_num < failing_step:
                step_gold = 'A'; step_gold_int = 1
            elif step_num == failing_step:
                step_gold = 'B'; step_gold_int = 0
            else:
                break

        conclusion = action_to_conclusion(action, domain)
        if conclusion is None:
            if step_gold == 'A':
                ns = apply_fn(state, action)
                if ns is not None: state = ns
            continue

        state_facts = [f.rstrip('.') for f in state.to_facts()]
        prompt = build_stepwise_prompt(rules, state_facts, conclusion)

        records.append({
            'split'      : split_label,
            'domain'     : domain,
            'plan_id'    : plan_id,
            'plan_label' : label,
            'fail_reason': fail_reason,
            'step_idx'   : step_idx,
            'step_num'   : step_num,
            'action'     : action,
            'conclusion' : conclusion,
            'state_facts': state_facts,
            'rules'      : rules,
            'goal_facts' : goal_facts,
            'prompt'     : prompt,
            'label'      : step_gold,
            'label_int'  : step_gold_int,
        })

        if step_gold == 'A':
            ns = apply_fn(state, action)
            if ns is not None: state = ns

    return records


def build_direct_for_plan(plan_id, row, domain, split_label):
    query = row.get('query','')
    gt    = row.get('ground_truth_plan','')
    parsed_query = parse_query(query, domain)
    parsed_gt    = parse_ground_truth(gt)
    if parsed_query is None: return None
    if parsed_gt['fail_reason'] == 'missing': return None

    label       = parsed_gt['label']
    fail_reason = parsed_gt['fail_reason']
    actions     = parsed_query['actions']
    rules       = parsed_query['rules']
    init_text   = parsed_query['init_text']
    goal_text   = parsed_query['goal_text']

    parse_facts_fn, parse_goal_fn, _ = get_domain_fns(domain)
    if domain in ('mystery','mystery_3'):
        init_state  = parse_facts_fn(abstract_mystery(init_text))
        goal_facts  = parse_goal_fn(abstract_mystery(goal_text))
        actions_abs = [abstract_mystery(a) for a in actions]
    else:
        init_state  = parse_facts_fn(init_text)
        goal_facts  = parse_goal_fn(goal_text)
        actions_abs = actions

    init_facts = [f.rstrip('.') for f in init_state.to_facts()]
    prompt = build_direct_prompt(rules, init_facts, goal_facts, actions_abs)
    gold   = 'A' if label == 1 else 'B'

    return {
        'split'      : split_label,
        'domain'     : domain,
        'plan_id'    : plan_id,
        'plan_label' : label,
        'fail_reason': fail_reason,
        'n_actions'  : len(actions),
        'actions'    : actions_abs,
        'rules'      : rules,
        'init_facts' : init_facts,
        'goal_facts' : goal_facts,
        'prompt'     : prompt,
        'label'      : gold,
        'label_int'  : label,
    }

print('Record builders: OK')

## 9. Build Split Manifest

Assigns each plan to `fewshot_invalid`, `fewshot_valid`, or `test`.

**Procedure per domain:**
1. Plans are separated into valid and invalid pools.
2. Each pool is shuffled with seed=42 for reproducibility.
3. Plans are selected in shuffled order with parseability verification,
   guaranteeing exactly 5 invalid + 5 valid parseable plans per domain.
4. All remaining plans are assigned to `test`.

The manifest is saved as `split_manifest.json` and loaded by all subsequent
notebooks (01a–01c, 02, 03, 04) to ensure consistent, reproducible splits.

In [ ]:
manifest    = {}   # {domain: {str(plan_id): 'fewshot_invalid'|'fewshot_valid'|'test'}}
split_stats = {}

for hf_domain, internal_domain in DOMAIN_MAP.items():
    rows = domain_groups.get(hf_domain, [])
    if not rows:
        print(f'[SKIP] {hf_domain}'); continue

    # Separate valid and invalid plan_ids
    valid_plan_ids   = []
    invalid_plan_ids = []
    for plan_id, row in enumerate(rows):
        gt = parse_ground_truth(row.get('ground_truth_plan'))
        if gt['label'] == 1: valid_plan_ids.append(plan_id)
        else:                 invalid_plan_ids.append(plan_id)

    # Shuffle with fixed seed
    rng = random.Random(SEED)
    shuffled_invalid = list(invalid_plan_ids); rng.shuffle(shuffled_invalid)
    shuffled_valid   = list(valid_plan_ids);   rng.shuffle(shuffled_valid)

    # Select first N that parse successfully — guarantees exactly N records
    fewshot_invalid_ids = []
    for pid in shuffled_invalid:
        if build_direct_for_plan(pid, rows[pid], internal_domain, 'check') is not None:
            fewshot_invalid_ids.append(pid)
        if len(fewshot_invalid_ids) == N_FEWSHOT_INVALID_PER_DOMAIN:
            break

    fewshot_valid_ids = []
    for pid in shuffled_valid:
        if build_direct_for_plan(pid, rows[pid], internal_domain, 'check') is not None:
            fewshot_valid_ids.append(pid)
        if len(fewshot_valid_ids) == N_FEWSHOT_VALID_PER_DOMAIN:
            break

    assert len(fewshot_invalid_ids) == N_FEWSHOT_INVALID_PER_DOMAIN,         f'{internal_domain}: only found {len(fewshot_invalid_ids)} parseable invalid plans'
    assert len(fewshot_valid_ids) == N_FEWSHOT_VALID_PER_DOMAIN,         f'{internal_domain}: only found {len(fewshot_valid_ids)} parseable valid plans'

    all_fewshot_ids = set(fewshot_invalid_ids) | set(fewshot_valid_ids)
    test_ids        = set(range(len(rows))) - all_fewshot_ids

    assert len(all_fewshot_ids & test_ids) == 0, f'OVERLAP in {internal_domain}!'

    domain_manifest = {}
    for pid in fewshot_invalid_ids: domain_manifest[str(pid)] = 'fewshot_invalid'
    for pid in fewshot_valid_ids:   domain_manifest[str(pid)] = 'fewshot_valid'
    for pid in test_ids:            domain_manifest[str(pid)] = 'test'
    manifest[internal_domain] = domain_manifest

    split_stats[internal_domain] = {
        'total_plans'            : len(rows),
        'valid_plans'            : len(valid_plan_ids),
        'invalid_plans'          : len(invalid_plan_ids),
        'fewshot_invalid_plans'  : len(fewshot_invalid_ids),
        'fewshot_valid_plans'    : len(fewshot_valid_ids),
        'test_plans'             : len(test_ids),
    }
    s = split_stats[internal_domain]
    print(f'{internal_domain:15s}: total={s["total_plans"]:4d} | '
          f'fewshot_inv={s["fewshot_invalid_plans"]:2d} '
          f'fewshot_val={s["fewshot_valid_plans"]:2d} '
          f'test={s["test_plans"]:4d}')

total_fewshot = sum(s['fewshot_invalid_plans']+s['fewshot_valid_plans']
                    for s in split_stats.values())
total_test    = sum(s['test_plans'] for s in split_stats.values())
print(f'\nTotal fewshot plans : {total_fewshot} '
      f'({N_FEWSHOT_INVALID_PER_DOMAIN*len(DOMAINS)} invalid + '
      f'{N_FEWSHOT_VALID_PER_DOMAIN*len(DOMAINS)} valid)')
print(f'Total test plans    : {total_test}')

## 10. Process All Domains

Iterates over all 5 domains and builds training and evaluation records.

**Stepwise records built:**
- `fewshot_invalid` plans → fewshot stepwise set (FlanT5 training in 01a, 01b)
- `test` plans → test stepwise set (evaluation in 02)
- `fewshot_valid` plans are excluded from stepwise records (used only for
  direct fine-tuning in 01c, not for stepwise training)

**Direct records built:**
- `test` plans → test direct set (evaluation in 02, 03, 04)
- Fewshot direct records for GPT 2-shot prompting are built separately in §11

In [ ]:
sw_fewshot_all = []   # stepwise records for fewshot_invalid plans (FlanT5 training)
sw_test_all    = []   # stepwise records for test plans
di_test_all    = []   # direct records for test plans
record_stats   = {}

for hf_domain, internal_domain in DOMAIN_MAP.items():
    rows = domain_groups.get(hf_domain, [])
    if not rows: continue

    dom_manifest = manifest[internal_domain]
    sw_fs, sw_te, di_te = [], [], []
    skip_sw, skip_di = 0, 0

    for plan_id, row in enumerate(rows):
        split_label = dom_manifest.get(str(plan_id), 'test')

        # Stepwise records: fewshot_invalid → training, test → evaluation
        # fewshot_valid plans are NOT added to stepwise fewshot (not used for stepwise training)
        if split_label in ('fewshot_invalid', 'test'):
            sw_recs = build_stepwise_for_plan(plan_id, row, internal_domain, split_label)
            if not sw_recs:
                skip_sw += 1
            elif split_label == 'fewshot_invalid':
                sw_fs.extend(sw_recs)
            else:
                sw_te.extend(sw_recs)

        # Direct records: test plans only
        if split_label == 'test':
            di_rec = build_direct_for_plan(plan_id, row, internal_domain, split_label)
            if di_rec is None: skip_di += 1
            else:              di_te.append(di_rec)

    sw_fewshot_all.extend(sw_fs)
    sw_test_all.extend(sw_te)
    di_test_all.extend(di_te)

    fs_cnt = Counter(r['label'] for r in sw_fs)
    te_cnt = Counter(r['label'] for r in sw_te)
    record_stats[internal_domain] = {
        'sw_fewshot': len(sw_fs), 'sw_test': len(sw_te), 'di_test': len(di_te),
        'fs_A': fs_cnt['A'], 'fs_B': fs_cnt['B'],
        'te_A': te_cnt['A'], 'te_B': te_cnt['B'],
        'skip_sw': skip_sw,  'skip_di': skip_di,
    }
    s = record_stats[internal_domain]
    print(f'{internal_domain:15s}: '
          f'sw_fewshot={s["sw_fewshot"]:4d}(A={s["fs_A"]} B={s["fs_B"]}) | '
          f'sw_test={s["sw_test"]:5d}(A={s["te_A"]} B={s["te_B"]}) | '
          f'di_test={s["di_test"]:4d}')

print(f'\nTotals:')
print(f'  Fewshot stepwise : {len(sw_fewshot_all)} records')
print(f'  Test    stepwise : {len(sw_test_all)} records')
print(f'  Test    direct   : {len(di_test_all)} records')

## 11. Build Direct Fewshot Records (GPT 2-Shot Examples)

Selects one valid and one invalid direct record per domain to use as
2-shot examples in GPT evaluation notebooks (03, 04).

The selected plans are the first (lowest plan_id) among the 5 `fewshot_valid`
and 5 `fewshot_invalid` plans per domain. All 10 records are saved to
`planbench_direct_fewshot.jsonl`. These plans are excluded from the test set
and never appear in evaluation results.

In [ ]:
di_fewshot_all = []   # direct records for GPT few-shot (1 valid + 1 invalid per domain)
gpt_fewshot_info = {}  # {domain: {'valid_plan_id': int, 'invalid_plan_id': int}}

for hf_domain, internal_domain in DOMAIN_MAP.items():
    rows = domain_groups.get(hf_domain, [])
    if not rows: continue

    dom_manifest = manifest[internal_domain]

    # Get fewshot_invalid and fewshot_valid plan_ids in selection order
    inv_ids = [int(pid) for pid, lbl in dom_manifest.items() if lbl == 'fewshot_invalid']
    val_ids = [int(pid) for pid, lbl in dom_manifest.items() if lbl == 'fewshot_valid']
    # Sort to get deterministic first element (manifest dict order not guaranteed)
    # Use the original shuffled order by selecting lowest plan_id as proxy
    # Actually we need the first-selected — stored in split_stats order
    # Safe: just use the first one that parses (they were all verified to parse)
    inv_pid = sorted(inv_ids)[0]   # deterministic: lowest plan_id among selected
    val_pid = sorted(val_ids)[0]

    inv_rec = build_direct_for_plan(inv_pid, rows[inv_pid], internal_domain, 'fewshot_invalid')
    val_rec = build_direct_for_plan(val_pid, rows[val_pid], internal_domain, 'fewshot_valid')

    assert inv_rec is not None, f'{internal_domain}: GPT invalid fewshot plan failed to parse'
    assert val_rec is not None, f'{internal_domain}: GPT valid fewshot plan failed to parse'
    assert inv_rec['label_int'] == 0, f'{internal_domain}: GPT invalid example is not invalid!'
    assert val_rec['label_int'] == 1, f'{internal_domain}: GPT valid example is not valid!'

    di_fewshot_all.extend([val_rec, inv_rec])  # valid first, then invalid
    gpt_fewshot_info[internal_domain] = {
        'valid_plan_id'  : val_pid,
        'invalid_plan_id': inv_pid,
    }
    print(f'{internal_domain:15s}: GPT valid_plan_id={val_pid} invalid_plan_id={inv_pid}')

print(f'\nTotal GPT fewshot direct records: {len(di_fewshot_all)} '
      f'(2 per domain × {len(DOMAINS)} domains)')

## 12. Zero-Overlap Verification

Four assertions confirm data integrity before saving:

1. No plan_id appears in both fewshot and test splits in the manifest.
2. No stepwise record (domain, plan_id, step_idx) appears in both the
   fewshot and test record sets.
3. All GPT fewshot plans are outside the test set.
4. Exactly 50 fewshot plans exist for FlanT5 direct fine-tuning
   (5 fewshot_invalid + 5 fewshot_valid × 5 domains = 50 total).

In [ ]:
# 1. No plan appears in both fewshot and test in manifest
for domain in DOMAINS:
    dom_manifest = manifest.get(domain, {})
    fs_pids  = {pid for pid,split in dom_manifest.items() if 'fewshot' in split}
    te_pids  = {pid for pid,split in dom_manifest.items() if split == 'test'}
    overlap  = fs_pids & te_pids
    assert len(overlap) == 0, f'PLAN OVERLAP in {domain}: {overlap}'

# 2. No stepwise record appears in both fewshot and test
fs_keys = {(r['domain'], r['plan_id'], r['step_idx']) for r in sw_fewshot_all}
te_keys = {(r['domain'], r['plan_id'], r['step_idx']) for r in sw_test_all}
assert len(fs_keys & te_keys) == 0, f'RECORD OVERLAP: {len(fs_keys & te_keys)} shared records!'

# 3. GPT fewshot plans are NOT in test set
for domain, info in gpt_fewshot_info.items():
    dom_manifest = manifest[domain]
    for pid_key in ('valid_plan_id', 'invalid_plan_id'):
        pid   = str(info[pid_key])
        label = dom_manifest.get(pid, 'MISSING')
        assert label != 'test', f'{domain} GPT fewshot plan {pid} is in test set!'

# 4. FlanT5_Direct_FT50: verify 50 parseable fewshot plans exist
total_direct_fewshot_train = 0
for domain in DOMAINS:
    dom_manifest = manifest.get(domain, {})
    inv_ids = [int(pid) for pid,lbl in dom_manifest.items() if lbl=='fewshot_invalid']
    val_ids = [int(pid) for pid,lbl in dom_manifest.items() if lbl=='fewshot_valid']
    assert len(inv_ids) == N_FEWSHOT_INVALID_PER_DOMAIN,         f'{domain}: expected {N_FEWSHOT_INVALID_PER_DOMAIN} fewshot_invalid, got {len(inv_ids)}'
    assert len(val_ids) == N_FEWSHOT_VALID_PER_DOMAIN,         f'{domain}: expected {N_FEWSHOT_VALID_PER_DOMAIN} fewshot_valid, got {len(val_ids)}'
    total_direct_fewshot_train += len(inv_ids) + len(val_ids)

assert total_direct_fewshot_train == 50,     f'Expected 50 FlanT5_Direct_FT50 training plans, got {total_direct_fewshot_train}'

print('Zero-overlap verification : PASSED')
print(f'GPT fewshot not in test   : PASSED')
print(f'FlanT5_Direct_FT50 plans  : {total_direct_fewshot_train} (must be 50) — PASSED')

## 13. Class Balance Report

Prints three summaries:
1. **Fewshot stepwise class balance** — A vs B label distribution in the
   FlanT5 stepwise training data (from `fewshot_invalid` plans)
2. **Test direct class balance** — valid vs invalid plan counts in the
   evaluation set per domain
3. **GPT fewshot plan IDs** — which plan_ids are used for 2-shot GPT prompting

In [ ]:
SEP = '─'*72
print('FEWSHOT STEPWISE CLASS BALANCE (FlanT5 training data)')
print(f'{"Domain":15s}  {"Steps":>6s}  {"A (valid steps)":>16s}  {"B (invalid steps)":>18s}')
print(SEP)
for d in DOMAINS:
    if d not in record_stats: continue
    s   = record_stats[d]
    tot = s['sw_fewshot']
    print(f'{d:15s}  {tot:>6d}  '
          f'{s["fs_A"]:>6d}({s["fs_A"]/max(1,tot)*100:>4.1f}%)  '
          f'{s["fs_B"]:>6d}({s["fs_B"]/max(1,tot)*100:>4.1f}%)')

print()
print('TEST DIRECT CLASS BALANCE (evaluation set)')
print(f'{"Domain":15s}  {"Plans":>6s}  valid / invalid')
print(SEP)
for d in DOMAINS:
    di_d    = [r for r in di_test_all if r['domain'] == d]
    n_valid = sum(1 for r in di_d if r['label_int'] == 1)
    n_inv   = sum(1 for r in di_d if r['label_int'] == 0)
    print(f'{d:15s}  {len(di_d):>6d}  valid={n_valid} invalid={n_inv}')

print()
print('GPT FEWSHOT EXAMPLES')
print(SEP)
for d in DOMAINS:
    info = gpt_fewshot_info.get(d,{})
    print(f'{d:15s}: valid_plan_id={info.get("valid_plan_id","?")}  '
          f'invalid_plan_id={info.get("invalid_plan_id","?")}')

## 14. Save All Files to Drive

Saves five output files to `paper2/data/` and verifies each by reloading
and checking record counts.

Once this notebook completes, all subsequent notebooks can be run
independently. They load their inputs from `paper2/data/` and save
their outputs (adapters, results) to `paper2/adapters/` and `paper2/results/`.

In [ ]:
# 1. Manifest
with open(MANIFEST_PATH,'w') as f:
    json.dump({
        'seed'                       : SEED,
        'n_fewshot_invalid_per_domain': N_FEWSHOT_INVALID_PER_DOMAIN,
        'n_fewshot_valid_per_domain'  : N_FEWSHOT_VALID_PER_DOMAIN,
        'manifest'                   : manifest,
        'split_stats'                : split_stats,
        'record_stats'               : record_stats,
        'gpt_fewshot_info'           : gpt_fewshot_info,
    }, f, indent=2)
print(f'Manifest          → {MANIFEST_PATH}')

# 2. Stepwise fewshot (FlanT5 training)
with open(SW_FEWSHOT_PATH,'w') as f:
    for r in sw_fewshot_all: f.write(json.dumps(r)+'\n')
print(f'SW fewshot        → {SW_FEWSHOT_PATH} ({len(sw_fewshot_all)} records)')

# 3. Direct fewshot (GPT 2-shot examples)
with open(DI_FEWSHOT_PATH,'w') as f:
    for r in di_fewshot_all: f.write(json.dumps(r)+'\n')
print(f'DI fewshot (GPT)  → {DI_FEWSHOT_PATH} ({len(di_fewshot_all)} records)')

# 4. Stepwise test
with open(SW_TEST_PATH,'w') as f:
    for r in sw_test_all: f.write(json.dumps(r)+'\n')
print(f'SW test           → {SW_TEST_PATH} ({len(sw_test_all)} records)')

# 5. Direct test
with open(DI_TEST_PATH,'w') as f:
    for r in di_test_all: f.write(json.dumps(r)+'\n')
print(f'DI test           → {DI_TEST_PATH} ({len(di_test_all)} records)')

# 6. Stats
with open(STATS_PATH,'w') as f:
    json.dump({'record_stats':record_stats,'split_stats':split_stats}, f, indent=2)
print(f'Stats             → {STATS_PATH}')

# Reload verification
assert len([json.loads(l) for l in open(SW_FEWSHOT_PATH)]) == len(sw_fewshot_all)
assert len([json.loads(l) for l in open(DI_FEWSHOT_PATH)]) == len(di_fewshot_all)
assert len([json.loads(l) for l in open(SW_TEST_PATH)])    == len(sw_test_all)
assert len([json.loads(l) for l in open(DI_TEST_PATH)])    == len(di_test_all)
print('\nReload check: PASSED')
print('\n=== DATA PREPARATION COMPLETE ===')
print(f'All files saved to {DATA_DIR}')